### Computing the GCD of N Numbers via a Small Linear Combination

By Bézout's identity, the GCD of a set of integers $\{n_1, \ldots, n_p\}$ can always be expressed as an integer linear combination $a_1 n_1 + \cdots + a_p n_p = \gcd(n_1, \ldots, n_p)$. The **LLL lattice reduction algorithm** gives us an efficient way to find both the GCD and a set of small coefficients $a_i$ simultaneously.

In [4]:
# LLL Algorithm Implementation following An_Introduction_to_Mathematical_Cryptography by Silverman, Figure 7.8

import numpy as np

# The Gram-Schmidt process is used to orthogonalize the basis vectors, which is essential for the LLL algorithm to check the Lovász condition and perform size reduction effectively.
# It must not return the orthonormal basis, but rather the orthogonalized basis, as the LLL algorithm relies on the lengths of these vectors to determine when to swap and reduce.
def gram_schmidt(B):
    B_star = np.zeros_like(B, dtype=float)
    B_star[0] = B[0]
    for i in range(1, len(B)):
        B_star[i] = B[i]
        for j in range(i):
            mu = np.dot(B[i], B_star[j]) / np.dot(B_star[j], B_star[j])
            B_star[i] -= mu * B_star[j]
    return B_star


# The LLL algorithm takes a basis B of a lattice and a parameter delta (which controls the "reduction" level, typically set to 0.75) and returns a reduced basis of the same lattice.
def LLL(B, delta=0.75):
    B_star = gram_schmidt(B)
    k = 1
    n = len(B)
    while k < n:
        for j in range(k-1, -1, -1):
            mu_kj = np.dot(B[k], B_star[j]) / np.dot(B_star[j], B_star[j]) # Projection of B[k] onto B_star[j], normalized by the length of B_star[j], as indicated in the foot note.
            B[k] -= round(mu_kj) * B[j] # Size Reduction
                
        # At this poing, we have reduced B[k] with respect to all previous vectors. Now we check the Lovász condition to decide if we need to swap B[k] and B[k-1].
        if np.dot(B_star[k], B_star[k]) >= (delta - ((np.dot(B[k], B_star[k-1]))/np.dot(B_star[k-1], B_star[k-1]))**2 ) * np.dot(B_star[k-1], B_star[k-1]):
            k += 1
        else:
            B[[k, k-1]] = B[[k-1, k]] # Swap B[k] and B[k-1]
            B_star = gram_schmidt(B)  
            k = max(k-1, 1)
    return B

#### Lattice construction

Given $p$ integers $n_1, \ldots, n_p$, we build a $p \times (p+1)$ basis matrix whose $i$-th row is the standard unit vector $e_i$ augmented with $n_i$ in the last column:

$$
B = \begin{pmatrix}
1 & 0 & \cdots & 0 & n_1 \\
0 & 1 & \cdots & 0 & n_2 \\
\vdots & & \ddots & & \vdots \\
0 & 0 & \cdots & 1 & n_p
\end{pmatrix}
$$

Any integer linear combination $\sum_i a_i \mathbf{b}_i$ produces the lattice vector $(a_1, \ldots, a_p,\; a_1 n_1 + \cdots + a_p n_p)$. In particular, the **shortest** such vector whose last entry is nonzero is a strong candidate for $\gcd(n_1,\ldots,n_p)$, with the first $p$ entries giving the Bézout coefficients.

> **Note:** LLL finds a *short* basis vector, not necessarily the shortest. It is guaranteed to succeed for typical inputs, but may fail for adversarial or very large numbers where the reduced basis still doesn't expose the GCD in the last coordinate.

In [5]:
def gcd(numbers):
    
    basis = np.zeros((len(numbers), len(numbers)+1))
    for i in range(0, len(numbers)):
        row = np.zeros(len(numbers)+1)
        row[i] = 1
        row[len(numbers)] = numbers[i]
        basis[i] = row
    
    reduced_basis = LLL(basis)
    #print(reduced_basis)
    
    for i in range(len(numbers)):
        if reduced_basis[i][len(numbers)] == 0:
            continue
        
        sum = 0
        for j in range(len(numbers)+1):
            if j != len(numbers):
               sum += reduced_basis[i][j] * numbers[j]
            else:
                gcd = reduced_basis[i][j]
                if sum ==  gcd:
                    return reduced_basis[i] if gcd > 0 else -1 * reduced_basis[i]
    return [0]*(len(numbers)+1)

#### Verification step

After LLL reduction, we scan the rows of the reduced basis for a row whose last entry $g \neq 0$ satisfies
$$\sum_{j=1}^{p} a_j \, n_j = g,$$
confirming that $g$ is indeed realised as a linear combination of the inputs (not just an artifact of floating-point arithmetic). If $g < 0$, the entire row is negated so the GCD is always returned as a positive number.

In [6]:
arr = [48, 18, 30, 12, 120]
lincomb = gcd(arr)
    
for i in range(len(lincomb)):
    if i != len(arr):
        print(f"{arr[i]} * ({lincomb[i]}) + ", end="")
    else:
        print(f"= {lincomb[i]}")

48 * (-0.0) + 18 * (1.0) + 30 * (-0.0) + 12 * (-1.0) + 120 * (-0.0) + = 6.0
